[![Open In Colab](https://github.com/minsung-kim-dev/Vectorizing-Raster-Maps/blob/main/notebooks/raster2seq_smoke_test.ipynb)](https://github.com/minsung-kim-dev/Vectorizing-Raster-Maps/blob/main/notebooks/raster2seq_smoke_test.ipynb)

# Raster2Seq stage-1 (shapes only) smoke test

Trains Raster2Seq for a few steps on a handful of OSM crops, to check that the data, model and loss work together. With so few images the model should start memorising them: the token-type loss (`loss_ce`) drops within a few passes, the coordinate loss (`loss_coords`) more slowly. This is a pipeline check, not a useful model.

Runs on Google Colab (GPU or CPU) or locally in the `raster2seq` env. On Colab: **Runtime → Change runtime type → GPU** is faster but optional.

What it does:
1. Installs Raster2Seq and its dependencies (Colab only)
2. Loads a small COCO dataset (`coco_osm_mini.zip`, made by `data_preprocess/create_coco_osm.py`)
3. Patches three things so Raster2Seq runs outside `main_ddp.py` (no multi-GPU setup, no wandb)
4. Builds the model with the stage-1 settings from Raster2Seq's `tools/train_cc5k.sh` (`--semantic_classes -1`)
5. Trains for a few steps and plots the loss

## 1. Settings

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    R2S = "/content/Raster2Seq"           # cloned below
    DATA = "/content/coco_osm_mini"       # unzipped from the uploaded coco_osm_mini.zip
else:
    R2S = os.path.abspath("../../Raster2Seq")
    DATA = os.path.abspath("../data/coco_osm_mini")

SPLIT = "train"      # which split to train on
N_IMAGES = 4         # how many images to use
BATCH_SIZE = 2
STEPS = 50           # passes over the N_IMAGES images
IMAGE_SIZE = 256     # 512 is closer to real training but ~4x slower
SEQ_LEN = 512        # shapes past this many tokens are dropped
NUM_BINS = 32        # grid size for position tokens
PRETRAINED = True    # ImageNet ResNet-50 weights (downloaded automatically, ~100 MB); False = random init

## 2. Install (Colab only)

Clones Raster2Seq and installs the packages its training code imports (Colab already has torch, numpy, opencv, shapely, …). On a GPU runtime it also compiles Raster2Seq's deformable-attention CUDA op; if that fails (or on CPU) step 4 falls back to Raster2Seq's plain-PyTorch version, which is slower but equivalent.

In [ ]:
import subprocess, torch

def sh(cmd):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)

if IN_COLAB:
    if not os.path.exists(R2S):
        sh(f"git clone -q https://github.com/Cornell-VAILab/Raster2Seq.git {R2S} && git -C {R2S} checkout -q a6c4e27")  # pinned, see README
    sh("pip install -q pycocotools descartes einops fvcore iopath omegaconf cloudpickle plotly scikit-image")
    if torch.cuda.is_available():
        try:
            sh(f"cd {R2S}/models/ops && python setup.py build install > build.log 2>&1")
        except subprocess.CalledProcessError:
            print(f"Compiling the CUDA op failed (see {R2S}/models/ops/build.log); using the PyTorch fallback.")

## 3. Data

On Colab, upload `coco_osm_mini.zip` from this repo's `data/` folder when prompted (or copy it from Google Drive). It's a 40-crop dataset in the layout Raster2Seq expects: `{train,val,test}/*.png` + `annotations/{split}.json`.

In [ ]:
if IN_COLAB and not os.path.exists(DATA):
    from google.colab import files
    files.upload()                       # choose coco_osm_mini.zip
    sh("unzip -q coco_osm_mini.zip -d /content")

print(sorted(os.listdir(DATA)), "|", len(os.listdir(f"{DATA}/{SPLIT}")), f"{SPLIT} images")

## 4. Patches

Three workarounds so Raster2Seq runs here without editing its code:
- **wandb** – `main_ddp.py` imports it for logging; a stub module avoids installing/configuring it.
- **Deformable attention** – the compiled CUDA op is used when available on a GPU; otherwise Raster2Seq's plain-PyTorch reference implementation (`ms_deform_attn_core_pytorch`).
- **Annotations edited in place** – Raster2Seq's loader resizes annotations inside its COCO object, so loading an image twice shrinks its shapes again. We hand it a copy each time.

In [ ]:
import types, copy, torch

sys.path.insert(0, R2S)
sys.modules.setdefault("wandb", types.ModuleType("wandb"))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
try:
    import MultiScaleDeformableAttention  # noqa: F401  (compiled CUDA op)
    use_pytorch_attn = device.type == "cpu"
except ImportError:
    sys.modules["MultiScaleDeformableAttention"] = types.ModuleType("MultiScaleDeformableAttention")
    use_pytorch_attn = True

from models.ops.functions import MSDeformAttnFunction, ms_deform_attn_func

if use_pytorch_attn:
    MSDeformAttnFunction.apply = staticmethod(
        lambda value, shapes, level_start, locations, weights, im2col_step:
        ms_deform_attn_func.ms_deform_attn_core_pytorch(value, shapes, locations, weights)
    )

if not PRETRAINED:
    import torchvision
    _resnet50 = torchvision.models.resnet50
    torchvision.models.resnet50 = lambda **kw: _resnet50(**{**kw, "pretrained": False})

print("device:", device, "| deformable attention:", "PyTorch fallback" if use_pytorch_attn else "compiled CUDA op")

## 5. Config

Stage-1 settings from Raster2Seq's `tools/train_cc5k.sh`, adapted to the OSM data:
- `--semantic_classes -1` – shapes only, no labels
- `--dataset_name waffle` – resize + pad to a square, keep all categories
- `--input_channels 3` – RGB maps
- `disable_image_transform` – no random flips/rotations, so the model can memorise the few images

In [ ]:
import argparse
from main_ddp import get_args_parser

args = argparse.ArgumentParser(parents=[get_args_parser()]).parse_args([
    "--dataset_name", "waffle", "--dataset_root", DATA,
    "--semantic_classes", "-1", "--input_channels", "3",
    "--poly2seq", "--seq_len", str(SEQ_LEN), "--num_bins", str(NUM_BINS), "--image_size", str(IMAGE_SIZE),
    "--batch_size", str(BATCH_SIZE), "--num_workers", "0", "--device", device.type,
    "--label_smoothing", "0.1", "--cls_loss_coef", "5", "--coords_loss_coef", "20",
    "--disable_poly_refine", "--dec_attn_concat_src", "--use_anchor", "--converter_version", "v3",
])
args.with_poly_refine = not args.disable_poly_refine   # main_ddp.py applies this after parsing
args.disable_image_transform = True   # not a main_ddp.py option, but the data loader reads it

## 6. Data loader

In [ ]:
from torch.utils.data import DataLoader, Subset
from datasets import build_dataset

dataset = build_dataset(image_set=SPLIT, args=args)
_load_anns = dataset.coco.loadAnns
dataset.coco.loadAnns = lambda ids: copy.deepcopy(_load_anns(ids))   # see patches above
args.vocab_size = dataset.get_vocab_size()

# the sequence tensors to batch together (same as main_ddp.py's collator)
SEQ_KEYS = ["delta_x1", "delta_x2", "delta_y1", "delta_y2", "seq11", "seq21", "seq12", "seq22",
            "target_seq", "token_labels", "mask", "target_polygon_labels"]

def collate(batch):
    return batch, {k: torch.stack([item.pop(k) for item in batch]) for k in SEQ_KEYS}

loader = DataLoader(Subset(dataset, range(min(N_IMAGES, len(dataset)))),
                    batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate, num_workers=0)
print(len(loader.dataset), "images,", len(loader), "batches per pass")

## 7. Model

In [ ]:
from models import build_model

model, criterion = build_model(args, tokenizer=dataset.get_tokenizer())
model.to(device)
criterion.to(device)
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                              lr=args.lr, weight_decay=args.weight_decay)
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parameters")

## 8. Train

Same step as Raster2Seq's `engine.train_one_epoch`: the decoder sees the true previous tokens, and the loss combines token type (corner / separator / end, `loss_ce`) and coordinates (`loss_coords`), weighted by `--cls_loss_coef` and `--coords_loss_coef`.

In [ ]:
import time

history = []
model.train(); criterion.train()
for step in range(STEPS):
    t0 = time.time()
    for batch, extras in loader:
        samples = [item["image"].to(device) for item in batch]
        extras = {k: v.to(device) for k, v in extras.items()}
        loss_dict = criterion(model(samples, extras), extras)
        weights = criterion.weight_dict
        loss = sum(loss_dict[k] * weights[k] for k in loss_dict if k in weights)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), args.clip_max_norm)
        optimizer.step()

        history.append({k: float(loss_dict[k]) for k in weights if k in loss_dict} | {"total": loss.item()})
    parts = ", ".join(f"{k} {v:.3f}" for k, v in history[-1].items())
    print(f"pass {step + 1}/{STEPS} ({time.time() - t0:.1f}s): {parts}")

## 9. Loss curve

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
for key in history[0]:
    ax.plot([h[key] for h in history], label=key)
ax.set(xlabel="training step", ylabel="loss", yscale="log", title="Training loss on a few images")
ax.legend()
plt.show()

## Next steps

- If training runs and `loss_ce` falls, the pipeline works end to end. For `loss_coords` to fall clearly, use `PRETRAINED = True` and more `STEPS`. Real training uses `main_ddp.py` via `torchrun` on a GPU machine (see Raster2Seq's `tools/train_*.sh` and `tools/finetune_*.sh`).
- Stage 2 (labels): `--semantic_classes` = number of labels + 1, plus `--per_token_sem_loss --jointly_train`.